# Exemplo 
Um experimento realizado para investigar o efeito do número de mols de cobalto (x1) e a temperatura de calcificação (x2) na área de superfície de um catalisador de hidróxido ferro-cobalto (y) gerou os dados a seguir (“Structural changes and surface properties of CoxFe3 _ xO4 Spinels”, J. of Che-mical Tech. and Biotech., 1994: 161-170). 



In [1]:
#@title Resposta a)
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf  #adicionada na semana 6


In [2]:
lstx1 = (0.6, 0.6, 0.6, 0.6, 0.6, 1.0, 1.0, 1.0,
         1.0, 1.0, 2.6, 2.6, 2.6, 2.6, 2.6, 2.8,
         2.8, 2.8, 2.8, 2.8)
lstx2 = (200, 250, 400, 500, 600, 200, 250, 400,
         500, 600, 200, 250, 400, 500, 600, 200,
         250, 400, 500, 600)
lstx3 = np.multiply(lstx1, lstx2)
lsty = (90.6, 82.7, 58.7, 43.2, 25.0, 127.1, 112.3,
        19.6, 17.8, 9.1, 53.1, 52.0, 43.4, 42.4, 31.6,
        40.9, 37.9, 27.5, 27.3, 19.0)
# Construir o DataFrame e nomear as colunas
df = pd.DataFrame(list(zip(lstx1, lstx2, lstx3, lsty)),
                  columns=["x1", "x2", "x3", "y"])
df.head(20)

,x1,x2,x3,y
0,0.6,200,120.0,90.6
1,0.6,250,150.0,82.7
2,0.6,400,240.0,58.7
3,0.6,500,300.0,43.2
4,0.6,600,360.0,25.0
5,1.0,200,200.0,127.1
6,1.0,250,250.0,112.3
7,1.0,400,400.0,19.6
8,1.0,500,500.0,17.8
9,1.0,600,600.0,9.1


In [3]:
regmul = smf.ols('y ~ x1 + x2 + x3', data=df)
res = regmul.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.780
Model:                            OLS   Adj. R-squared:                  0.739
Method:                 Least Squares   F-statistic:                     18.92
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           1.64e-05
Time:                        11:43:29   Log-Likelihood:                -82.063
No. Observations:                  20   AIC:                             172.1
Df Residuals:                      16   BIC:                             176.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    185.4857     21.197      8.750      0.0

In [4]:
#coeficientes estimados 
b = res.params
b0 = b.iloc[0]
b1 = b.iloc[1]
b2 = b.iloc[2]
b3 = b.iloc[3]
print('y = {0}  {1} x1  {2} x2 + {3} x1x2 '.format(b0, b1, b2, b3))

y = 185.48573955525512  -45.969465970352765 x1  -0.3015028638814051 x2 + 0.08880138140162638 x1x2 


In [5]:
#@title Resposta b)
auxi = df.iloc[10:15, :]
auxi.head()

,x1,x2,x3,y
10,2.6,200,520.0,53.1
11,2.6,250,650.0,52.0
12,2.6,400,1040.0,43.4
13,2.6,500,1300.0,42.4
14,2.6,600,1560.0,31.6


In [6]:
resi = res.resid
df1 = pd.DataFrame(list(zip(lstx1, lstx2, lstx3, lsty, resi)),
                   columns=["x1", "x2", "x3", "y", "e"])
auxi2 = df1.iloc[10:15, :]
auxi2.head()

,x1,x2,x3,y,e
10,2.6,200,520.0,53.1,1.258726
11,2.6,250,650.0,52.0,3.689690
12,2.6,400,1040.0,43.4,5.682581
13,2.6,500,1300.0,42.4,11.744508
14,2.6,600,1560.0,31.6,8.006435


In [7]:
#@title Resposta d)
import scipy.stats

F = res.fvalue
k = res.df_model  # grau do modelo
n = res.nobs  # num. amostras
dfn = k
dfd = n - (k + 1)
alpha = 0.05  #nível de confiança.
F_critico = scipy.stats.f.ppf(1 - alpha, dfn, dfd)
print("F_crit=", F_critico)  #tabela F-dist

F_crit= 3.2388715174535863


In [8]:
#@title Resposta e)
#usar a tabela tstudent pata t
from scipy.stats import t

alpha = 0.05  # significia = 5%
df = n - (k + 1)  # graus de liberdade
v = t.ppf(1 - alpha / 2, df)
tt = v
print(f't_crit=: {v}')

t_crit=: 2.1199052992212546


In [9]:
#@title complementares
#ANOVA
table = sm.stats.anova_lm(res, typ=1)  #
print(table)

            df       sum_sq      mean_sq          F    PR(>F)
x1         1.0  2384.155795  2384.155795   8.890838  0.008808
x2         1.0  9562.712520  9562.712520  35.660642  0.000020
x3         1.0  3276.659972  3276.659972  12.219106  0.002991
Residual  16.0  4290.539713   268.158732        NaN       NaN


In [10]:
#ANOVA
table = sm.stats.anova_lm(res, typ=1)  # typ pode ser 2 ou 3
totalSM = np.sum(table.iloc[0:3, 1])
totalMean = np.sum(table.iloc[0:3, 1]) / np.sum(table.iloc[0:3, 0])
print(table.iloc[3, 0:4])
print("SQE=", totalSM)
print(totalMean)

df           16.000000
sum_sq     4290.539713
mean_sq     268.158732
F                  NaN
Name: Residual, dtype: float64
SQE= 15223.528287230456
5074.509429076818


In [11]:
s2 = totalSM / (n - (k + 1))
s = pow(s2, 1 / 2)
s2, s

(np.float64(951.4705179519035), np.float64(30.845915741827206))